# Notebook 2 — The While Language Syntax

**Covers Chapter 1 of the notes.** We zoom in on the BNF grammars (arithmetic expressions, boolean expressions, statements), walk all 6 worked examples from the chapter, and solve exercises 1–10.

## What you'll be able to do by the end

- Read any BNF grammar fluently — including the While grammar in §1.5.
- Tell the difference between concrete and abstract syntax.
- Predict whether a given While program is syntactically valid.
- Spot grammar ambiguity and explain *why* the grammar is ambiguous.
- Write a correct While program for any of the chapter's exercise problems.

## Active mode reminder

🎯 PREDICT cells: don't run the next cell until you've filled in your guess. The point is to commit to an answer, then check.

In [1]:
import sys
from pathlib import Path

# Make this work whether jupyter was started from notebooks/ or from the repo root.
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("could not find while_lang.py — start jupyter from the notebooks/ folder")

from while_lang import (
    parse, run, trace,
    A, B, step, step_iter,
    Config, Transition, StepBudgetExceeded,
    check_state, check_steps,
)

## §1. What BNF grammars say (and don't say)

BNF (Backus-Naur Form) is a way of writing down a language's syntax — *which strings of symbols are valid programs*. It says nothing about what those programs *do*; that's semantics, and we cover it in N3.

The basic shape:

```
<thing> ::= <alternative-1> | <alternative-2> | <alternative-3>
```

Read this as: **a `<thing>` is one of these alternatives**. The `|` is "or". Brackets like `<thing>` mark *non-terminals* — categories that get further defined elsewhere. Things that aren't bracketed are *terminals* — literal symbols you type.

BNF is **recursive**: alternatives can mention `<thing>` itself. For example, the chapter writes:

```
<aexp> ::= <num> | <var> | <aexp> + <aexp> | <aexp> − <aexp> | <aexp> × <aexp>
```

An arithmetic expression is *either* a numeral, *or* a variable, *or* the sum of two arithmetic expressions, *or* the difference of two, *or* the product of two. That's a recursive definition: the base cases are `<num>` and `<var>`; the step cases are the three binary operations.

**To check whether a string is in the language**, you have to find a sequence of rule applications (a *derivation*) that produces that string. If you can build the parse tree, the string is valid. If you can't, it isn't.

We don't have to build derivations by hand — that's what the parser is for. But understanding what the grammar *says* is essential for the exam.

## §2. Arithmetic expressions (AExp)

From the chapter:

```
<aexp> ::= <num> | <var> | <aexp> + <aexp> | <aexp> − <aexp> | <aexp> × <aexp>
```

The set generated by this grammar is called **AExp**. Examples of arithmetic expressions:

- `7` — a numeral.
- `x` — a variable.
- `x + 1` — sum of `x` and `1`.
- `(x + 1) × y` — product of `(x + 1)` and `y`. The brackets aren't formally part of the grammar, but the chapter allows them for readability.
- `7 − 4 − 2` — difference, but ambiguous (see Exercise 2).

Things **not** in AExp:

- `x / y` — division isn't a primitive in While.
- `x²` — neither is exponentiation.
- `x + tt` — `tt` is a boolean, not an arithmetic value. The grammar wouldn't generate this.

Let's see the parser at work — convert some strings to ASTs and confirm the parser agrees with our reading of the grammar:

In [2]:
# Wrap each AExp in 'x := ...' so it parses as a complete program; pull out the .expr.
for src in ['7', 'x', 'x + 1', '(x + 1) * y', '7 - 4 - 2']:
    ast = parse(f'tmp := {src}').expr
    print(f'{src!r:25} → {ast}')

'7'                       → Num(value=7)
'x'                       → Var(name='x')
'x + 1'                   → Add(left=Var(name='x'), right=Num(value=1))
'(x + 1) * y'             → Mul(left=Add(left=Var(name='x'), right=Num(value=1)), right=Var(name='y'))
'7 - 4 - 2'               → Sub(left=Sub(left=Num(value=7), right=Num(value=4)), right=Num(value=2))


## Exercise 2(a) — ambiguous arithmetic expressions

> Why is the grammar for While ambiguous? Give an example of an ambiguous arithmetic expression.

**Why ambiguous:** the rule `<aexp> ::= <aexp> + <aexp>` lets the parser choose either of two structures for `7 − 4 − 2`:

- **Left-associative reading:** `(7 − 4) − 2 = 1`
- **Right-associative reading:** `7 − (4 − 2) = 5`

The grammar doesn't tell you which one is the right parse — both are derivable from the rules. So the same string has two distinct parse trees, and (worse) they evaluate to different values.

**A concrete ambiguous AExp**: `7 - 4 - 2`. Or `1 + 2 + 3` (both readings give 6, but the parse trees still differ — same string, two derivations).

**How real languages handle this**: by adopting an associativity convention (e.g. `−` and `+` left-associative). Lark's grammar in `while_lang.py` does exactly this — the rule `aexp: aexp "+" mul_term` is left-recursive, which makes Lark pick the left-associative parse.

In [3]:
# Our parser picks the left-associative reading.
ast = parse('z := 7 - 4 - 2').expr
print(ast)
# Should print Sub(left=Sub(left=Num(7), right=Num(4)), right=Num(2)) — i.e. (7-4)-2.

# And evaluating it confirms the result:
print('value:', A(ast, {}))   # (7-4)-2 = 1

Sub(left=Sub(left=Num(value=7), right=Num(value=4)), right=Num(value=2))
value: 1


## §3. Boolean expressions (BExp)

From the chapter:

```
<bexp> ::= ff | tt | <aexp> = <aexp> | <aexp> ≤ <aexp> | ¬<bexp> | <bexp> ∧ <bexp>
```

ASCII translation in our interpreter: `<=` for `≤`, `!` for `¬`, `&` for `∧`.

Note what's **missing**:

- No `<` (strict less-than). Workaround: `a <= b ∧ ¬(a = b)`.
- No `>` or `>=`. Workaround: swap operands of `<=`, e.g. `a ≥ b` becomes `b <= a`.
- No `∨` (or). See Exercise 1.
- No `→` (implication). See Exercise 1.

These omissions are deliberate. Theoretical languages are kept lean to make proofs shorter — every connective you add becomes another case to consider when proving things by induction over the grammar.

## Exercise 1 — extending boolean expressions

> Assume `a` and `b` are boolean expressions. (a) How can we represent `a ∨ b` in While? (b) How can we represent `a → b`?

### (a) Disjunction `a ∨ b`

By **De Morgan's law**: `a ∨ b ≡ ¬(¬a ∧ ¬b)`. 

- If both `a` and `b` are false, then `¬a ∧ ¬b` is true, and `¬(¬a ∧ ¬b)` is false. ✓
- If either is true, then `¬a ∧ ¬b` is false, and `¬(¬a ∧ ¬b)` is true. ✓

In our ASCII syntax: `!((!a) & (!b))`.

### (b) Implication `a → b`

Implication is true exactly when it's not the case that `a` is true and `b` is false. So `a → b ≡ ¬(a ∧ ¬b)`.

Or equivalently `(¬a) ∨ b`, which using (a) becomes `¬(a ∧ ¬b)` again — same answer.

ASCII: `!(a & !b)`.

In [4]:
# Verify Exercise 1(a): a ∨ b ≡ ¬(¬a ∧ ¬b)
# Use 'a' and 'b' as concrete boolean expressions: a = (x = 0), b = (y = 0).
# Truth table over all four combinations.
for x in [0, 1]:
    for y in [0, 1]:
        sigma = {'x': x, 'y': y}
        original = (x == 0) or (y == 0)                              # a ∨ b
        encoded = B(parse('if !((!(x = 0)) & (!(y = 0))) then (z := 1) else (z := 0)').cond, sigma)
        print(f'x={x}, y={y}: a∨b={original},  encoded={encoded},  {"✓" if original == encoded else "✗"}')

x=0, y=0: a∨b=True,  encoded=True,  ✓
x=0, y=1: a∨b=True,  encoded=True,  ✓
x=1, y=0: a∨b=True,  encoded=True,  ✓
x=1, y=1: a∨b=False,  encoded=False,  ✓


In [5]:
# Verify Exercise 1(b): a → b ≡ ¬(a ∧ ¬b)
for x in [0, 1]:
    for y in [0, 1]:
        sigma = {'x': x, 'y': y}
        original = (not (x == 0)) or (y == 0)        # a → b ≡ (¬a) ∨ b
        encoded = B(parse('if !((x = 0) & (!(y = 0))) then (z := 1) else (z := 0)').cond, sigma)
        print(f'x={x}, y={y}: a→b={original},  encoded={encoded},  {"✓" if original == encoded else "✗"}')

x=0, y=0: a→b=True,  encoded=True,  ✓
x=0, y=1: a→b=False,  encoded=False,  ✓
x=1, y=0: a→b=True,  encoded=True,  ✓
x=1, y=1: a→b=True,  encoded=True,  ✓


## Exercise 2(b) — ambiguous boolean expression

Same problem as 2(a): the rule `<bexp> ::= <bexp> ∧ <bexp>` allows either grouping for `a ∧ b ∧ c`. Both are derivable; the grammar doesn't disambiguate.

Concrete example: `tt ∧ ff ∧ tt` — left-associative gives `(tt ∧ ff) ∧ tt = ff`, right-associative gives `tt ∧ (ff ∧ tt) = ff`. Same value (∧ is associative), but two distinct parse trees from the same string.

## §4. Statements

From the chapter:

```
<stmt> ::= <var> := <aexp> | skip | <stmt>;<stmt> |
           if <bexp> then <stmt> else (<stmt>) |
           while <bexp> do (<stmt>)
```

Five forms:

1. **Assignment** — `x := a`. Replace the value of variable `x` with the result of arithmetic expression `a`.
2. **Skip** — `skip`. Does nothing. ("Theoretical reasons" per the chapter — we'll see why in N3.)
3. **Sequential composition** — `S1; S2`. Do `S1` first, then `S2`.
4. **Conditional** — `if b then S1 else (S2)`. Evaluate `b`; if true, execute `S1`; else execute `S2`.
5. **Loop** — `while b do (S)`. While `b` is true, execute `S`; check `b` again; repeat.

**On brackets:** the grammar requires brackets around the body of `if-else` and `while-do`. Without them, programs are ambiguous (see chapter §1.8). When written across multiple lines, indentation is allowed instead, but in a single line we use brackets.

## Exercise 2(c) — ambiguous statement

**The chapter's own example:** `S1; S2; S3`. By the rule `<stmt> ::= <stmt>;<stmt>`, this can be derived as either `(S1; S2); S3` or `S1; (S2; S3)`.

Both readings produce the same execution behaviour (we'll prove this in N4 — the *associativity of sequential composition*, Proposition 3). But the parse trees differ — same string, two derivations.

Section 1.8 of the chapter discusses this in more detail and notes that the issue is harmless for `;` (because of associativity), but real for `if` and `while` bodies (which is why brackets are required there).

## §5. The chapter's worked examples

Now we walk all 6 examples from chapter 1, running each through the interpreter to confirm what they do.

### Example 1 — division and remainder

Compute `m div n` (in `d`) and `m mod n` (in `r`).

In [6]:
div_prog = '''
    r := m;
    d := 0;
    while n <= r do (
        d := d + 1;
        r := r - n
    )
'''

# m=10, n=3 → d=3, r=1.
print(run(div_prog, {'m': 10, 'n': 3}))

# m=20, n=7 → d=2, r=6.
print(run(div_prog, {'m': 20, 'n': 7}))

{'m': 10, 'n': 3, 'r': 1, 'd': 3}
{'m': 20, 'n': 7, 'r': 6, 'd': 2}


### Example 2 — find an integer solution to `ax² + bx + c = 0`

The chapter's program loops over `x ∈ ℕ`, testing whether either `+x` or `−x` is a solution. The flag `y` is set to 1 when a solution is found.

**Bug worth noticing:** as written this loops forever if no solution exists in ℤ (e.g. `a=1, b=−2, c=2`, the chapter's Example 10). We use `max_steps` to cap the search.

We also need `b × x` to be expressed using only `+`, `−`, `×`. The chapter writes `a × x × x − b × x + c = 0` for the `−x` case — that's how `a · (−x)² + b · (−x) + c` simplifies. Verify:

- `a · (−x)² + b · (−x) + c = a·x² − b·x + c`. ✓

In [7]:
quad_prog = '''
    x := 0; y := 0;
    while !(y = 1) do (
        if a * x * x + b * x + c = 0 then
            y := 1
        else (
            if a * x * x - b * x + c = 0 then
                y := 1
            else (
                skip
            )
        );
        x := x + 1
    )
'''

# x² − 5x + 6 = 0 has roots 2 and 3 in ℕ. Should find x=2 first.
# After finding the solution at x=2, the loop body still increments x at the bottom.
# So x ends up at 3 (the next value after the one that worked).
print(run(quad_prog, {'a': 1, 'b': -5, 'c': 6}))

{'a': 1, 'b': -5, 'c': 6, 'x': 3, 'y': 1}


In [8]:
# x² − 2x + 2 = 0 has no real roots → the program runs forever.
# Bound it with max_steps to demonstrate.
try:
    run(quad_prog, {'a': 1, 'b': -2, 'c': 2}, max_steps=200)
except StepBudgetExceeded as e:
    print(f'caught: {e}')

caught: step budget exceeded after 200 steps — likely non-terminating


### Example 3 — gcd by repeated subtraction

Subtract the smaller from the larger until they're equal.

In [9]:
# Note: the chapter writes 'if y < x' but we don't have <; substitute 'y <= x ∧ ¬(y = x)' or use '!(x <= y)'.
# '!(x <= y)' means y < x (under the equality of integers).
gcd_prog = '''
    x := m; y := n;
    while !(x = y) do (
        if !(x <= y) then
            x := x - y
        else (
            y := y - x
        )
    )
'''

# gcd(12, 30) = 6
print(run(gcd_prog, {'m': 12, 'n': 30}))

# gcd(48, 210) = 6  (the chapter's Example 11)
print(run(gcd_prog, {'m': 48, 'n': 210}))

{'m': 12, 'n': 30, 'x': 6, 'y': 6}
{'m': 48, 'n': 210, 'x': 6, 'y': 6}


### Example 4 — primality test

Walk all candidate divisors `y` from `n−1` down to `2`. For each, see if `y` divides `n`. If any does, set `z := 0` (not prime). The inner `while` computes `n mod y` by subtraction (the same pattern as Example 1).

In [10]:
prime_prog = '''
    y := n - 1;
    z := 1;
    while 2 <= y & z = 1 do (
        r := n;
        while y <= r do (
            r := r - y
        );
        if r = 0 then
            z := 0
        else (
            skip
        );
        y := y - 1
    )
'''

for n in [2, 3, 4, 7, 9, 11, 15, 17]:
    final = run(prime_prog, {'n': n})
    z = final.get("z", 0)
    print(f'n = {n}: z = {z}  ({"prime" if z == 1 else "composite"})')

n = 2: z = 1  (prime)
n = 3: z = 1  (prime)
n = 4: z = 0  (composite)
n = 7: z = 1  (prime)
n = 9: z = 0  (composite)
n = 11: z = 1  (prime)
n = 15: z = 0  (composite)
n = 17: z = 1  (prime)


### Example 5 — integer square root

Find the largest `z` with `z² ≤ x`. Initial guess `z = 0`, increment until `(z+1)² > x`.

In [11]:
sqrt_prog = '''
    z := 0;
    while (z + 1) * (z + 1) <= x do (
        z := z + 1
    );
    r := x - z * z
'''

for x in [0, 1, 3, 9, 10, 16, 17, 100]:
    final = run(sqrt_prog, {'x': x})
    print(f'x = {x}: z = {final.get("z", 0)}, r = {final.get("r", 0)}  (z² + r = {final.get("z", 0)**2 + final.get("r", 0)})')

x = 0: z = 0, r = 0  (z² + r = 0)
x = 1: z = 1, r = 0  (z² + r = 1)
x = 3: z = 1, r = 2  (z² + r = 3)
x = 9: z = 3, r = 0  (z² + r = 9)
x = 10: z = 3, r = 1  (z² + r = 10)
x = 16: z = 4, r = 0  (z² + r = 16)
x = 17: z = 4, r = 1  (z² + r = 17)
x = 100: z = 10, r = 0  (z² + r = 100)


### Example 6 — nth Fibonacci

Iteratively walk the Fibonacci sequence using a temporary variable. The variable `y` holds the current term, `z` holds the previous term, `t` is a swap helper.

In [12]:
fib_prog = '''
    y := 1;
    z := 0;
    while 1 <= x do (
        t := z;
        z := y;
        y := t + y;
        x := x - 1
    )
'''

# F(0) = 0, F(1) = 1, F(2) = 1, F(3) = 2, F(4) = 3, F(5) = 5, F(6) = 8, F(7) = 13.
# Per the chapter's table, after running with x=4 we get z=3 (=F(4)).
for n in range(8):
    final = run(fib_prog, {'x': n})
    print(f'n = {n}: F(n) = {final.get("z", 0)}')

n = 0: F(n) = 0
n = 1: F(n) = 1
n = 2: F(n) = 1


n = 3: F(n) = 2
n = 4: F(n) = 3
n = 5: F(n) = 5
n = 6: F(n) = 8
n = 7: F(n) = 13


## §6. Exercises 3–10

Now the exercises. We've already covered 1 and 2 above.

## Exercise 3 — repeat-until in While (chapter has solution)

> Different concepts for loops exist. Explain how we may implement a 'repeat until' loop in While.

**The trick:** `repeat S until b` runs `S` *first*, then checks `b` — opposite of `while b do S`, which checks `b` first.

**Solution** (this matches the chapter's official answer):

```
S;                     # run the body once unconditionally
while !b do (S)        # then keep running it until b becomes true
```

After `S` runs once, the `while !b` loop checks `b`. If `b` is true, the loop exits → repeat ran exactly once. If `b` is false, the body `S` runs again, and the cycle repeats.

In [13]:
# Demo: 'repeat (x := x + 1) until x = 5' starting with x = 0.
# Should run the body 5 times → x = 5.
repeat_until = '''
    x := x + 1;
    while !(x = 5) do (
        x := x + 1
    )
'''
print(run(repeat_until, {'x': 0}))   # x = 5

{'x': 5}


## Exercise 4 — `if` is unnecessary

> Show that we don't need `if` statements in While.

**Goal:** simulate `if b then S else (S')` using only assignment, sequential composition, and `while` loops. We need a way to make exactly one of `S` or `S'` run, depending on `b`.

**The trick:** use a flag variable `done` to ensure each branch runs at most once.

```
done := 0;
while b & done = 0 do (
    S;
    done := 1
);
while !(done = 1) & !b do (
    S';
    done := 1
)
```

**Why it works** (case-by-case):

- `b` is initially **true** → first loop's body runs once: `S` executes, `done := 1`. Next iteration: condition is `b & done = 0` = `b & ff` = `ff`, loop exits. Second loop: condition is `¬(done = 1) & ¬b` = `¬tt & ...` = `ff`, body doesn't fire. ✓ Only `S` ran.
- `b` is initially **false** → first loop's body doesn't fire. Second loop: condition is `¬(done = 1) & ¬b` = `tt & tt` = `tt`. Body runs once: `S'` executes, `done := 1`. Next iteration: `¬(done = 1)` is `ff`, loop exits. ✓ Only `S'` ran.

**Subtle point:** even if `S` modifies the value of `b`, the first loop has already exited (because `done = 1` makes the AND false), so the second loop's `¬b` check is irrelevant — it's gated by `¬(done = 1)`, which is also false.

Combined with Exercise 3 we have the result that **While needs only assignment, `;`, and `while`** to express everything `if` and repeat-until could express. We rigorously establish this connects to the operational semantics in Exercise 15 (covered in N4).

In [14]:
# Demo: simulate 'if x = 0 then (y := 1) else (y := 2)' without using if.
no_if = '''
    done := 0;
    while x = 0 & done = 0 do (
        y := 1;
        done := 1
    );
    while !(done = 1) & !(x = 0) do (
        y := 2;
        done := 1
    )
'''

# x = 0: y should be 1.
print(run(no_if, {'x': 0}))
# x = 99: y should be 2.
print(run(no_if, {'x': 99}))

{'done': 1, 'y': 1}
{'x': 99, 'done': 1, 'y': 2}


## Exercise 5 — even/odd, then `n mod m`

> Write a While program that takes integer `n` and determines whether it's even or odd. Then use it to compute `n mod m`.

### Part 1: even/odd

Decrement `n` by 1 repeatedly, flipping a parity flag each time, until `n` reaches 0. The flag at the end is `n mod 2`.

Edge case: if `n` is negative, we negate first. Without that handling we'd report "even" for any negative number.

We store the result in `is_odd`: 0 if even, 1 if odd.

In [15]:
parity_prog = '''
    is_odd := 0;
    m := n;
    if m <= -1 then
        m := 0 - m
    else (
        skip
    );
    while 1 <= m do (
        m := m - 1;
        is_odd := 1 - is_odd
    )
'''

for n in [-5, -2, -1, 0, 1, 2, 3, 4, 5, 10]:
    final = run(parity_prog, {'n': n})
    print(f'n = {n:3}: is_odd = {final.get("is_odd", 0)}')

n =  -5: is_odd = 1
n =  -2: is_odd = 0
n =  -1: is_odd = 1
n =   0: is_odd = 0
n =   1: is_odd = 1
n =   2: is_odd = 0
n =   3: is_odd = 1
n =   4: is_odd = 0
n =   5: is_odd = 1
n =  10: is_odd = 0


### Part 2: `n mod m`

Same trick, but instead of subtracting 1 each iteration we subtract `m`. We assume `m > 0` (otherwise the operation isn't well-defined for our purposes).

If `n` is negative, we keep adding `m` until it's non-negative. Then we keep subtracting `m` while it's still ≥ `m`. The final value is `n mod m`.

In [16]:
# Compute n mod m, assuming m > 0. Result in r.
mod_prog = '''
    r := n;
    while r <= -1 do (
        r := r + m
    );
    while m <= r do (
        r := r - m
    )
'''

for (n, m) in [(10, 3), (20, 7), (15, 5), (-7, 3), (0, 4)]:
    final = run(mod_prog, {'n': n, 'm': m})
    expected = n % m
    actual = final.get('r', 0)
    print(f'{n} mod {m} = {actual}  (Python: {expected},  match: {actual == expected})')

10 mod 3 = 1  (Python: 1,  match: True)


20 mod 7 = 6  (Python: 6,  match: True)
15 mod 5 = 0  (Python: 0,  match: True)
-7 mod 3 = 2  (Python: 2,  match: True)
0 mod 4 = 0  (Python: 0,  match: True)


## Exercise 6 — count cubes in `[m, n]`

> Let `m` and `n` be natural numbers. Write a program that finds the number of natural numbers `x` with the property that `x` is a cube and `m ≤ x ≤ n`.

**Approach:** walk `i = 0, 1, 2, ...` while `i³ ≤ n`. For each `i`, if `m ≤ i³`, increment a counter.

In [17]:
count_cubes = '''
    count := 0;
    i := 0;
    while i * i * i <= n do (
        if m <= i * i * i then
            count := count + 1
        else (
            skip
        );
        i := i + 1
    )
'''

# Cubes in [0, 30]: 0, 1, 8, 27 → 4.
print(run(count_cubes, {'m': 0, 'n': 30}))

# Cubes in [10, 200]: 27, 64, 125 → 3.
print(run(count_cubes, {'m': 10, 'n': 200}))

# Cubes in [1, 1]: just 1 → 1.
print(run(count_cubes, {'m': 1, 'n': 1}))

{'n': 30, 'count': 4, 'i': 4}
{'m': 10, 'n': 200, 'count': 3, 'i': 6}
{'m': 1, 'n': 1, 'count': 1, 'i': 2}


## Exercise 7 — solve `mx + ny = k` for natural `x, y` (chapter has solution)

> Write a program that finds a solution `x, y ∈ ℕ` if one exists.

**Approach (chapter's solution):** enumerate pairs `(x, y)` with `0 ≤ y ≤ x` in lexicographic order. For each pair, check whether `mx + ny = k` OR (by swapping) `my + nx = k`. If we find a solution by swapping, we put the swapped pair into `(x, y)`.

Pairs explored: `(0,0), (1,0), (1,1), (2,0), (2,1), (2,2), (3,0), ...`

In [18]:
diophantine = '''
    x := 0; y := 0;
    while !(m * x + n * y = k) do (
        x := x + 1;
        y := 0;
        while !(m * x + n * y = k) & y <= x & !(y = x) do (
            if m * y + n * x = k then (
                z := x;
                x := y;
                y := z
            ) else (
                y := y + 1
            )
        )
    )
'''

# 3x + 4y = 7 → (x=1, y=1) is a natural solution.
print(run(diophantine, {'m': 3, 'n': 4, 'k': 7}))

{'m': 3, 'n': 4, 'k': 7, 'x': 1, 'y': 1}


### Exercise 7 part (b) — extend to integer solutions

> Explain how you might extend your program so that it will find an integer solution if one exists.

**Approach:** for any candidate `(x, y) ∈ ℕ²`, also test the four sign combinations `(±x, ±y)` and the swapped `(±y, ±x)`. Eight nested if-statements; if any matches, store the signed pair.

We don't run this — the chapter only asks for an explanation, and the implementation is mechanical.

## Exercises 8 and 9 — functions β, β', φ, φ⁻¹

These reference Sections 6.2.1 and 6.2.2 of the course notes, which are **not in the materials provided here.** They cover encoding/decoding pairs of natural numbers as single naturals (β-style indexing functions, φ-style pairing function).

Without those definitions I can't write the programs faithfully. **Verify with a GTA** for the exact specifications before attempting these in earnest. Standard candidates from the literature include:

- **β: ℤ → ℕ** typically defined as `2n` if `n ≥ 0` else `2(−n)−1`. Inverse `β⁻¹` recovers the integer from the natural.
- **φ: ℕ × ℕ → ℕ** the Cantor pairing function: `φ(m, n) = (m + n)(m + n + 1)/2 + n`.

If your version of these is the standard one, the programs are mechanical to write — multiplications, divisions (by repeated subtraction), conditional sign-flipping. Skip in the absence of the section text.

## Exercise 10 — Euclid's algorithm

> Write a program that implements Euclid's algorithm to find `gcd(m, n)`. Output in `x`.

**Standard Euclidean algorithm** (using mod, *not* repeated subtraction): `gcd(a, b) = gcd(b, a mod b)`, terminating when `b = 0`. Then `gcd = a`.

We implement `a mod b` inline using the subtraction loop pattern from Example 1.

In [19]:
euclid = '''
    x := m;
    y := n;
    while !(y = 0) do (
        r := x;
        while y <= r do (
            r := r - y
        );
        x := y;
        y := r
    )
'''

import math
for (m, n) in [(48, 18), (12, 30), (100, 75), (17, 5), (1, 1), (12, 0)]:
    final = run(euclid, {'m': m, 'n': n})
    expected = math.gcd(m, n)
    print(f'gcd({m}, {n}) = {final.get("x", 0)}  (Python: {expected})')

gcd(48, 18) = 6  (Python: 6)
gcd(12, 30) = 6  (Python: 6)
gcd(100, 75) = 25  (Python: 25)


gcd(17, 5) = 1  (Python: 1)
gcd(1, 1) = 1  (Python: 1)
gcd(12, 0) = 12  (Python: 12)


## Summary

You've now seen:

- The full BNF grammar of While (AExp, BExp, statements).
- Why the grammar is ambiguous (rules like `<aexp> ::= <aexp> + <aexp>` permit two parse trees for `1 + 2 + 3`).
- All 6 worked chapter examples, executed and verified.
- Exercises 1–10 with explanations, except 8 and 9 (which depend on material not provided).

Next: **N3 — semantics**. We zoom in on chapter 2 §2.1–2.4: states, the `A` and `B` evaluators, the small-step rules. Examples 7–11 from the chapter become *interpreter runs* — and Exercise 14 (the showpiece) is a complete formal trace of the Diophantine program.